# 第1章：Agent 基础概念

本章介绍 AI Agent 的核心概念、基本结构和运行原理。你将学习 Agent 的四大特征（自主性、反应性、主动性、社交性），理解感知-决策-执行的循环架构，并通过代码实现一个可运行的简单 Agent。

**核心知识点**：
- Agent 的定义与核心特征
- 感知 → 推理 → 行动（Perception → Reasoning → Action）循环
- ReAct（Reasoning + Acting）模式
- 状态管理与记忆
- 环境交互与反馈

## 学习目标与环境准备

**学习目标**：
1. 理解 Agent 的核心概念和架构模式
2. 掌握 Agent 的基本编程实现
3. 能够构建简单的自主 Agent

**环境准备**：本章仅依赖 Python 标准库，无需额外安装。

## 1.1 Agent 核心类定义

Agent 是具有自主决策能力的智能实体。下面定义一个基础 Agent 类，包含名称、状态、记忆和动作空间等核心属性。

In [ ]:
from abc import ABC, abstractmethod
from dataclasses import dataclass, field
from typing import Any, Callable, Dict, List, Optional
from enum import Enum
import random


class AgentState(Enum):
    IDLE = "idle"
    THINKING = "thinking"
    ACTING = "acting"
    WAITING = "waiting"
    DONE = "done"


@dataclass
class Perception:
    raw_input: str
    structured: Dict[str, Any] = field(default_factory=dict)
    timestamp: float = 0.0


@dataclass
class Action:
    name: str
    params: Dict[str, Any] = field(default_factory=dict)


class BaseAgent(ABC):
    def __init__(self, name: str):
        self.name = name
        self.state = AgentState.IDLE
        self.memory: List[Dict[str, Any]] = []
        self.actions: Dict[str, Callable] = {}

    def perceive(self, observation: str) -> Perception:
        perception = Perception(raw_input=observation)
        self.memory.append({"type": "perception", "content": perception})
        print(f"[{self.name}] 感知到: {observation}")
        return perception

    @abstractmethod
    def reason(self, perception: Perception) -> Action:
        pass

    def act(self, action: Action) -> str:
        self.state = AgentState.ACTING
        if action.name in self.actions:
            result = self.actions[action.name](**action.params)
        else:
            result = f"执行动作: {action.name} with {action.params}"
        self.memory.append({"type": "action", "content": action, "result": result})
        self.state = AgentState.IDLE
        print(f"[{self.name}] 执行: {action.name} → {result}")
        return str(result)

    def register_action(self, name: str, func: Callable):
        self.actions[name] = func
        print(f"[{self.name}] 注册动作: {name}")

    def step(self, observation: str) -> str:
        perception = self.perceive(observation)
        action = self.reason(perception)
        return self.act(action)


print("Agent 基础类定义完成 ✓")
print(f"可用动作状态: {[s.value for s in AgentState]}")

## 1.2 Agent 四大特征演示

Agent 具备四大核心特征：**自主性**（无外部干预自行决策）、**反应性**（感知环境变化并快速响应）、**主动性**（主动采取目标导向行为）、**社交性**（与其他 Agent 或人类交互协作）。以下代码演示这四大特征。

In [ ]:
class FeatureDemoAgent(BaseAgent):
    def __init__(self, name: str, goals: List[str]):
        super().__init__(name)
        self.goals = goals
        self.internal_timer = 0

    def reason(self, perception: Perception) -> Action:
        self.state = AgentState.THINKING
        input_text = perception.raw_input.lower()

        if "help" in input_text:
            return Action(name="assist", params={"user": perception.raw_input})
        elif "weather" in input_text:
            return Action(name="check_weather", params={"location": "local"})
        elif "greet" in input_text or "hello" in input_text:
            return Action(name="greet", params={"message": "Hello from agent!"})
        else:
            return Action(name="analyze", params={"input": perception.raw_input})

    def proactive_act(self) -> Action:
        self.internal_timer += 1
        if self.internal_timer % 5 == 0:
            return Action(name="report_status", params={"goal_progress": len(self.memory)})
        return Action(name="idle", params={})

    def negotiate(self, other_agent: str, proposal: str) -> str:
        self.state = AgentState.THINKING
        response = f"协同响应 to {other_agent}: agree to {proposal}"
        self.memory.append({"type": "social", "with": other_agent, "result": response})
        print(f"[{self.name}] 社交互动 with {other_agent}: {response}")
        return response


agent = FeatureDemoAgent(name="DemoAgent", goals=["收集信息", "协助用户"])

agent.register_action("greet", lambda **kw: f"打招呼: {kw.get('message', 'Hello')}")
agent.register_action("analyze", lambda **kw: f"分析: {kw.get('input', '')}")
agent.register_action("assist", lambda **kw: f"协助用户: {kw.get('user', '')}")
agent.register_action("check_weather", lambda **kw: f"天气查询: {kw.get('location', 'unknown')}")
agent.register_action("report_status", lambda **kw: f"状态报告: 进度={kw.get('goal_progress', 0)}")
agent.register_action("idle", lambda **kw: "空闲中")

print("=== 自主性与反应性演示 ===")
agent.step("Hello, I need help")
agent.step("What is the weather today?")
agent.step("Tell me about AI")

print("=== 主动性演示 ===")
for _ in range(6):
    action = agent.proactive_act()
    if action.name != "idle":
        print(f"[主动行为] {action.name}: {action.params}")

print("=== 社交性演示 ===")
agent.negotiate("Agent_B", "共享数据资源")
agent.negotiate("Human_User", "任务优先级的调整")

## 1.3 感知-决策-执行循环

PID（Perception → Inference → Decision → Execution）循环是 Agent 运行的核心模式。每次循环 Agent 从环境收集信息（感知），进行推理判断（决策），然后执行相应动作（执行），并接收反馈进入下一轮循环。

In [ ]:
import time


class PIDLoopAgent(BaseAgent):
    def __init__(self, name: str):
        super().__init__(name)
        self.knowledge_base: Dict[str, str] = {
            "python": "Python 是一种解释型、面向对象的高级编程语言。",
            "agent": "AI Agent 是能够自主感知环境并执行动作以实现目标的智能实体。",
            "rl": "强化学习是智能体通过与环境的交互来学习最优策略的方法。",
        }

    def reason(self, perception: Perception) -> Action:
        self.state = AgentState.THINKING
        text = perception.raw_input.lower()
        for keyword, knowledge in self.knowledge_base.items():
            if keyword in text:
                return Action(name="answer", params={"answer": knowledge})
        if "?" in text or "what" in text or "how" in text:
            return Action(name="search", params={"query": perception.raw_input})
        return Action(name="acknowledge", params={"input": perception.raw_input})


agent = PIDLoopAgent(name="PID_Agent")
agent.register_action("answer", lambda answer: answer)
agent.register_action(
    "search",
    lambda query: f"在知识库中未找到关于 '{query}' 的信息，将进行外部查询。",
)
agent.register_action(
    "acknowledge",
    lambda input: f"已收到: '{input}'，但无法采取有效行动。",
)

print("=== PID 循环运行 ===")
observations = ["什么是Python?", "告诉我Agent的定义", "现在几点了？"]
for i, obs in enumerate(observations, 1):
    print(f"--- 循环 {i} ---")
    time.sleep(0.2)
    result = agent.step(obs)
    print(f"结果: {result}")

print(f"记忆条目数: {len(agent.memory)}")

## 1.4 ReAct 循环基础

ReAct（Reasoning + Acting）是当前最主流的 Agent 推理范式。Agent 交替进行思考（Thought）、行动（Action）和观察（Observation），形成闭环的推理-执行链。下面实现一个基础的 ReAct 循环框架。

In [ ]:
class ReActAgent(BaseAgent):
    def __init__(self, name: str, max_steps: int = 5):
        super().__init__(name)
        self.max_steps = max_steps
        self.tools = {
            "calculator": lambda expr: f"计算结果: {eval(expr)}",
            "search": lambda q: f"搜索结果: 关于'{q}'的模拟结果",
            "translate": lambda text, target: f"'{text}' 翻译为{target}",
        }

    def think(self, observation: str) -> str:
        if "计算" in observation or "+" in observation or "*" in observation:
            return "需要使用 calculator 工具进行计算"
        elif "搜索" in observation or "查找" in observation:
            return "需要使用 search 工具搜索信息"
        elif "翻译" in observation:
            return "需要使用 translate 工具翻译内容"
        else:
            return "可以直接回答，无需工具"

    def reason(self, perception: Perception) -> Action:
        thought = self.think(perception.raw_input)
        if "calculator" in thought:
            expr = perception.raw_input.replace("计算", "").strip()
            return Action(name="calculator", params={"expr": expr if expr else "1+1"})
        elif "search" in thought:
            return Action(name="search", params={"q": perception.raw_input})
        elif "translate" in thought:
            return Action(name="translate", params={"text": perception.raw_input, "target": "英文"})
        return Action(name="direct_answer", params={"response": perception.raw_input})

    def react_loop(self, task: str) -> str:
        print(f"[{self.name}] 开始 ReAct 循环，任务: {task}")
        observation = task
        for step in range(1, self.max_steps + 1):
            print(f"--- Step {step} ---")
            thought = self.think(observation)
            print(f"  Thought: {thought}")
            perception = self.perceive(observation)
            action = self.reason(perception)
            print(f"  Action: {action.name}({action.params})")
            result = self.act(action)
            print(f"  Observation: {result}")
            observation = result
            if "直接" in thought or action.name == "direct_answer":
                print("任务完成!")
                return result
        return f"达到最大步数 {self.max_steps}"


agent = ReActAgent(name="ReAct_Bot")
agent.register_action("calculator", lambda expr: f"计算结果: {eval(expr)}")
agent.register_action("search", lambda q: f"搜索结果: 关于'{q}'的模拟结果")
agent.register_action("translate", lambda text, target="英文": f"'{text}' 翻译为{target}: [翻译结果]")
agent.register_action("direct_answer", lambda response: f"直接回答: {response}")

print("=== ReAct 循环演示 1 ===")
agent.react_loop("计算 3 + 5 * 2 的结果")

print("=== ReAct 循环演示 2 ===")
agent2 = ReActAgent(name="ReAct_Bot_2")
agent2.register_action("search", lambda q: f"搜索结果: 关于'{q}'的模拟结果")
agent2.register_action("direct_answer", lambda response: f"直接回答: {response}")
agent2.react_loop("搜索AI Agent的最新进展")

## 1.5 环境交互模拟

Agent 总是运行在某个环境中。下面构建一个简单的网格环境，Agent 需要通过移动来达到目标位置，展示了 Agent 与环境之间的感知-行动循环。

In [ ]:
class GridEnvironment:
    def __init__(self, width: int = 5, height: int = 5):
        self.width = width
        self.height = height
        self.agent_pos = (0, 0)
        self.target_pos = (width - 1, height - 1)
        self.obstacles = {(2, 2), (3, 1)}

    def get_observation(self) -> str:
        x, y = self.agent_pos
        tx, ty = self.target_pos
        nearby = self._get_nearby()
        return (
            f"位置({x},{y}) 目标({tx},{ty}) "
            f"周围: {nearby}"
        )

    def _get_nearby(self) -> Dict[str, str]:
        x, y = self.agent_pos
        directions = {
            "上": (x, y - 1),
            "下": (x, y + 1),
            "左": (x - 1, y),
            "右": (x + 1, y),
        }
        result = {}
        for dname, (nx, ny) in directions.items():
            if not (0 <= nx < self.width and 0 <= ny < self.height):
                result[dname] = "边界"
            elif (nx, ny) in self.obstacles:
                result[dname] = "障碍物"
            else:
                result[dname] = "可通行"
        return result

    def move(self, direction: str) -> str:
        x, y = self.agent_pos
        moves = {"上": (0, -1), "下": (0, 1), "左": (-1, 0), "右": (1, 0)}
        if direction not in moves:
            return f"无效方向: {direction}"
        dx, dy = moves[direction]
        nx, ny = x + dx, y + dy
        if not (0 <= nx < self.width and 0 <= ny < self.height):
            return "撞到边界!"
        if (nx, ny) in self.obstacles:
            return "撞到障碍物!"
        self.agent_pos = (nx, ny)
        return f"移动到 ({nx}, {ny})"

    def display(self):
        grid = []
        for y in range(self.height):
            row = []
            for x in range(self.width):
                if (x, y) == self.agent_pos:
                    row.append("A")
                elif (x, y) == self.target_pos:
                    row.append("G")
                elif (x, y) in self.obstacles:
                    row.append("#")
                else:
                    row.append(".")
            grid.append(" ".join(row))
        print("".join(grid))


print("=== 初始网格环境 ===")
env = GridEnvironment(5, 5)
env.display()
print(f"{env.get_observation()}")

print("=== 模拟移动 ===")
moves_sequence = ["右", "右", "下", "下", "右", "右", "下", "下"]
for move in moves_sequence:
    result = env.move(move)
    print(f"{move} → {result}")
    if env.agent_pos == env.target_pos:
        print("🎯 到达目标!")
        break

print("=== 最终网格状态 ===")
env.display()

## 1.6 状态管理与工作记忆

Agent 需要在多步推理中维护状态。以下实现一个带工作记忆的 Agent，能够记住历史交互并在后续步骤中利用这些信息。

In [ ]:
class StatefulAgent(BaseAgent):
    def __init__(self, name: str):
        super().__init__(name)
        self.working_memory: Dict[str, Any] = {}
        self.interaction_count = 0
        self.context: List[Dict[str, str]] = []

    def update_state(self, key: str, value: Any):
        self.working_memory[key] = value
        print(f"[{self.name}] 更新状态 {key} = {value}")

    def recall(self, key: str, default: Any = None) -> Any:
        return self.working_memory.get(key, default)

    def reason(self, perception: Perception) -> Action:
        self.state = AgentState.THINKING
        self.interaction_count += 1
        text = perception.raw_input

        self.context.append({"role": "user", "content": text})
        context_summary = f"已有 {len(self.context)} 轮对话，记忆状态: {list(self.working_memory.keys())}"

        if "名字" in text or "我叫" in text:
            name = text.split("我叫")[-1].strip().rstrip("。")
            if "我叫" in text:
                self.update_state("user_name", name)
                return Action(name="greet_personal", params={"name": name})
        if "喜欢" in text or "爱好" in text:
            self.update_state(
                "topic", text.replace("我喜欢", "").replace("我的爱好是", "").strip()
            )
        user_name = self.recall("user_name", "用户")
        return Action(name="respond", params={"name": user_name, "context": context_summary})


agent = StatefulAgent(name="记忆Agent")
agent.register_action(
    "greet_personal", lambda name: f"你好 {name}! 很高兴认识你，我已经记住了你的名字。"
)
agent.register_action(
    "respond",
    lambda name, context: f"{name}，我收到了你的消息。({context})",
)

print("=== 有状态交互 ===")
agent.step("你好，我叫小明")
agent.step("我喜欢编程")
agent.step("你还记得我的名字吗？")

print(f"工作记忆内容: {agent.working_memory}")
print(f"对话上下文轮数: {len(agent.context)}")

## 1.7 多 Agent 协作基础

在实际应用中，多个 Agent 可以协同工作完成复杂任务。下面演示两个 Agent 通过消息传递进行协作。

In [ ]:
class Message:
    def __init__(self, sender: str, receiver: str, content: str, msg_type: str = "info"):
        self.sender = sender
        self.receiver = receiver
        self.content = content
        self.msg_type = msg_type

    def __repr__(self):
        return f"[{self.msg_type}] {self.sender} → {self.receiver}: {self.content}"


class CollaborativeAgent(BaseAgent):
    def __init__(self, name: str, role: str):
        super().__init__(name)
        self.role = role
        self.inbox: List[Message] = []
        self.outbox: List[Message] = []
        self.partner: Optional["CollaborativeAgent"] = None

    def connect(self, other: "CollaborativeAgent"):
        self.partner = other
        other.partner = self

    def send(self, content: str, msg_type: str = "info"):
        if self.partner:
            msg = Message(self.name, self.partner.name, content, msg_type)
            self.outbox.append(msg)
            self.partner.inbox.append(msg)
            print(f"  📤 {msg}")

    def process_inbox(self):
        while self.inbox:
            msg = self.inbox.pop(0)
            print(f"  📥 [{self.name}] 收到: {msg.content}")
            self.memory.append({"type": "message", "from": msg.sender, "content": msg.content})

    def reason(self, perception: Perception) -> Action:
        self.state = AgentState.THINKING
        content = perception.raw_input
        if self.role == "分析师":
            analysis = f"分析结果: {content} → 重要发现XYZ"
            if self.partner:
                self.send(analysis)
            return Action(name="report", params={"result": analysis})
        elif self.role == "执行者":
            exec_result = f"执行计划: 基于 {content} 分配任务"
            return Action(name="execute", params={"result": exec_result})
        return Action(name="ack", params={})


analyst = CollaborativeAgent(name="分析师Agent", role="分析师")
executor = CollaborativeAgent(name="执行者Agent", role="执行者")
analyst.connect(executor)

analyst.register_action("report", lambda result: result)
executor.register_action("execute", lambda result: result)
executor.register_action("ack", lambda: "确认收到")

print("=== 协作任务 ===")
analyst.step("分析用户需求数据")
executor.process_inbox()

print(f"分析师记忆条目: {len(analyst.memory)}")
print(f"执行者记忆条目: {len(executor.memory)}")
print(f"分析师发件箱: {len(analyst.outbox)} 条消息")
print(f"执行者收件箱: {len(executor.inbox)} 条消息")

## 1.8 完整 Agent 集成示例

将以上所学概念整合为一个完整的任务型 Agent。该 Agent 能够接收用户任务、规划步骤、选择工具执行并汇总结果。

In [ ]:
import json


class TaskAgent(BaseAgent):
    def __init__(self, name: str):
        super().__init__(name)
        self.task_history: List[Dict[str, Any]] = []
        self.tools = {
            "web_search": lambda q: f"搜索 '{q}' 返回: [模拟搜索结果1, 模拟搜索结果2]",
            "read_file": lambda path: f"文件 '{path}' 的内容: [模拟文件内容]",
            "write_file": lambda path, content: f"已写入文件 '{path}'",
            "send_email": lambda to, body: f"已发送邮件至 {to}",
            "calculate": lambda expr: f"{expr} = {eval(expr)}",
        }
        self.register_action("use_tool", self._use_tool)
        self.register_action("report", self._report)

    def _use_tool(self, tool_name: str, **params) -> str:
        if tool_name in self.tools:
            return self.tools[tool_name](**params)
        return f"未知工具: {tool_name}"

    def _report(self, summary: str) -> str:
        return f"任务报告: {summary}"

    def plan_task(self, task: str) -> List[Action]:
        plan = []
        if "搜索" in task or "查找" in task or "了解" in task:
            plan.append(Action(name="use_tool", params={"tool_name": "web_search", "q": task}))
        if "计算" in task:
            expr = task.split("计算")[-1].strip()
            plan.append(Action(name="use_tool", params={"tool_name": "calculate", "expr": expr if expr else "0"}))
        if "文件" in task or "读写" in task:
            plan.append(Action(name="use_tool", params={"tool_name": "read_file", "path": "/data/report.txt"}))
        plan.append(Action(name="report", params={"summary": f"完成: {task}"}))
        return plan

    def reason(self, perception: Perception) -> Action:
        if not hasattr(self, "_plan"):
            self._plan = self.plan_task(perception.raw_input)
            self._plan_index = 0
        if self._plan_index < len(self._plan):
            action = self._plan[self._plan_index]
            self._plan_index += 1
            return action
        return Action(name="report", params={"summary": "所有步骤已完成"})

    def execute_task(self, task: str):
        print(f"{'='*50}")
        print(f"任务: {task}")
        print(f"{'='*50}")
        if hasattr(self, "_plan"):
            del self._plan
        perception = Perception(raw_input=task)
        plan = self.plan_task(task)
        print(f"📋 规划了 {len(plan)} 个步骤:")
        for i, action in enumerate(plan, 1):
            print(f"  步骤{i}: {action.name}({action.params})")
        print(f"🔧 开始执行...")
        results = []
        for i, action in enumerate(plan, 1):
            print(f"  [{i}/{len(plan)}] {action.name}...", end=" ")
            result = self.act(action)
            results.append(result)
        task_record = {
            "task": task,
            "steps": len(plan),
            "results": results,
        }
        self.task_history.append(task_record)
        print(f"✅ 任务完成! 共执行 {len(plan)} 步")
        return results


agent = TaskAgent(name="TaskMaster")

agent.execute_task("搜索 AI Agent 最新发展")
agent.execute_task("计算 100 * 25 + 50")
agent.execute_task("查找最新的研究报告并搜索相关技术")

print(f"📊 历史任务总结:")
for i, record in enumerate(agent.task_history, 1):
    print(f"  任务{i}: {record['task'][:30]}... ({record['steps']}步骤)")

## 练习

1. **扩展 Agent 工具集**：为 `TaskAgent` 添加至少 3 个新工具（如数据库查询、图像处理、API调用等）。

2. **实现记忆衰减机制**：修改 `StatefulAgent`，使旧的记忆条目随着时间推移权重降低，并实现自动清理过期记忆。

3. **多 Agent 协商**：扩展 `CollaborativeAgent`，实现三个 Agent 的协商投票机制，当两个 Agent 对任务有不同意见时，通过投票达成一致。

4. **ReAct 循环增强**：为 `ReActAgent` 添加错误恢复机制，当工具调用失败时自动重试或切换备选工具。